<a href="https://colab.research.google.com/github/Jouzou-M/RAG-Email-Chatbot/blob/main/IMDB_Ratings_Sentiment_Analysis_NLP_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print('Running on TPU:', tpu.master())
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
except ValueError:
    strategy = tf.distribute.get_strategy()
    print('Running on CPU/GPU')


In [ ]:
MODEL_PATH = "sentiment_model_outputs"
os.makedirs(MODEL_PATH, exist_ok=True)

def prepare_dataset(max_features=10000, max_len=250):
    """
    Load and preprocess the IMDB dataset for sentiment analysis.
    Returns preprocessed training and testing data along with the tokenizer.
    """
    dataset = load_dataset("imdb")

    train_texts = dataset['train']['text']
    train_labels = np.array(dataset['train']['label'], dtype=np.float32)
    test_texts = dataset['test']['text']
    test_labels = np.array(dataset['test']['label'], dtype=np.float32)

    tokenizer = Tokenizer(num_words=max_features, oov_token='<OOV>')
    tokenizer.fit_on_texts(train_texts)

    train_sequences = tokenizer.texts_to_sequences(train_texts)
    test_sequences = tokenizer.texts_to_sequences(test_texts)

    train_padded = pad_sequences(train_sequences, maxlen=max_len, truncating='post', padding='post')
    test_padded = pad_sequences(test_sequences, maxlen=max_len, truncating='post', padding='post')

    return train_padded, train_labels, test_padded, test_labels, tokenizer

def create_dataset(X, y, batch_size=32, shuffle=True):
    """Convert data arrays into a tf.data.Dataset pipeline"""
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(X))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

def create_sentiment_model(vocab_size, max_len):
    """Build Model within TPU strategy scope"""
    with strategy.scope():
        model = models.Sequential([
            layers.Embedding(vocab_size, 64, input_length=max_len),
            layers.Bidirectional(layers.LSTM(32)),
            layers.Dense(16, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(1, activation='sigmoid')
        ])
        model.compile(
            optimizer=optimizers.Adam(learning_rate=1e-3),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
    return model

def manual_train_test_split(X, y, validation_split=0.2):
    """Manual Train-Validation Split with shuffling"""
    split_idx = int(len(X) * (1 - validation_split))
    indices = np.random.permutation(len(X))
    X_shuffled = X[indices]
    y_shuffled = y[indices]
    X_train = X_shuffled[:split_idx]
    y_train = y_shuffled[:split_idx]
    X_val = X_shuffled[split_idx:]
    y_val = y_shuffled[split_idx:]
    return X_train, y_train, X_val, y_val

def train_sentiment_model():
    """Main training function"""
    max_features = 10000
    max_len = 250

    train_padded, train_labels, test_padded, test_labels, tokenizer = prepare_dataset(
        max_features=max_features,
        max_len=max_len
    )

    X_train, y_train, X_val, y_val = manual_train_test_split(train_padded, train_labels)

    train_dataset = create_dataset(X_train, y_train, batch_size=32, shuffle=True)
    val_dataset = create_dataset(X_val, y_val, batch_size=32, shuffle=False)
    test_dataset = create_dataset(test_padded, test_labels, batch_size=32, shuffle=False)

    model = create_sentiment_model(vocab_size=max_features, max_len=max_len)

    checkpoint_path = os.path.join(MODEL_PATH, "best_model.keras")
    callbacks = [
        ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_accuracy'),
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    ]

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=5,
        callbacks=callbacks
    )

    test_loss, test_accuracy = model.evaluate(test_dataset)
    print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")

    model.save(os.path.join(MODEL_PATH, "final_sentiment_model.keras"))
    import pickle
    with open(os.path.join(MODEL_PATH, 'tokenizer.pickle'), 'wb') as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

    return model, tokenizer, history

def predict_sentiment(texts, model=None, tokenizer=None, max_len=250):
    """Predict sentiment for provided texts"""
    if model is None:
        model = tf.keras.models.load_model(os.path.join(MODEL_PATH, "final_sentiment_model.keras"))
    if tokenizer is None:
        import pickle
        with open(os.path.join(MODEL_PATH, 'tokenizer.pickle'), 'rb') as handle:
            tokenizer = pickle.load(handle)

    sequences = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences(sequences, maxlen=max_len, truncating='post', padding='post')
    predictions = model.predict(padded)
    return predictions

def main():
    global trained_model, tokenizer
    trained_model, tokenizer, history = train_sentiment_model()
    print("\nModel training completed!")

    sample_texts = [
        "This movie was absolutely fantastic! I loved every minute of it.",
        "What a terrible waste of time. I couldn't even finish watching it.",
        "Honestly don't know what to think, didn't understand the plot but i loved the 60s production and direction"
    ]

    predictions = predict_sentiment(sample_texts, trained_model, tokenizer)

    print("\nSentiment Analysis Results:")
    print("-" * 80)
    for text, pred in zip(sample_texts, predictions):
        sentiment = "Positive" if pred[0] > 0.5 else "Negative"
        confidence = pred[0] if sentiment == "Positive" else (1 - pred[0])
        print(f"Text: {text}")
        print(f"Sentiment: {sentiment}")
        print(f"Confidence: {confidence:.1%}")
        print("-" * 80)

if __name__ == "__main__":
    main()

